# Практическая работа. Кластеризация методом k средних
## «Найти группы, когда ответов нет»

**Цель:** написать метод k средних (алгоритм Ллойда) на чистом Python, увидеть, как он сходится и где застревает, реализовать повторные запуски и инициализацию k-means++, выбрать число кластеров методом «локтя» и проверить, найдет ли метод три сорта вина, если скрыть от него ответы.

> **Правило работы.** Внутри своих функций – только списки, словари, циклы, `math` и `random`. Библиотека `sklearn` нужна только для загрузки Wine и сверки в задании С8, `matplotlib` – только в готовых ячейках с графиками.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом – подсказка написана в комментарии `# <--`. Напечатанные значения должны совпасть с тем, что стоит после слова «ожидается». Готовые ячейки просто выполните. Сначала прочитайте теорию.

**План работы**

| Задание | Что пишем | Главный результат |
|---|---|---|
| С1 | `dist2`, `closest_center`, `assign_clusters` | шаг назначения |
| С2 | `update_centers` | шаг пересчета центров |
| С3 | `inertia` | мера качества кластеризации |
| С4 | `kmeans` | алгоритм Ллойда на сцене из шести объектов |
| С5 | `random_init`, `kmeans_restarts` | локальные минимумы и повторные запуски |
| С6 | `kmeans_pp_init` | умная инициализация k-means++ |
| С7 | метод «локтя» | число кластеров |
| С8 | `contingency`, `purity` | Wine без ответов: кластеры против сортов |

## Теория: метод k средних

**1. Задача кластеризации.** До сих пор у нас были ответы $y$: сорт вина, класс объекта на дороге. Это обучение с учителем. В **кластеризации** ответов нет: есть только объекты $x_1, \dots, x_n$, и нужно разбить их на группы (**кластеры**) так, чтобы внутри группы объекты были похожи, а между группами различались. Это обучение без учителя (Б1.1, виды обучения). Примеры: беспилотник разделяет облако точек лидара на отдельные объекты; магазин находит группы покупателей; фотограф сокращает число цветов картинки.

Задача поставлена нечетко: единственно правильного ответа нет. Гудфеллоу (раздел 5.8.2) приводит пример: фотографии красных и серых грузовиков и легковушек можно разбить на два кластера и «по типу», и «по цвету», и оба разбиения разумны. Какое из них нужно нам, алгоритм не знает.

**2. Идея метода k средних.** Каждый кластер $j$ описывается своим **центром** $\mu_j$ – точкой в пространстве признаков. Объект относится к кластеру ближайшего центра. Хорошие центры – те, до которых объектам в сумме близко. Мера качества – **инерция** (сумма квадратов расстояний от объектов до центров их кластеров):

$$J = \sum_{i=1}^{n} \left\| x_i - \mu_{c_i} \right\|^2, \qquad \left\| a - b \right\|^2 = \sum_{d} (a_d - b_d)^2,$$

где $c_i$ – номер кластера объекта $i$. Число кластеров $k$ – гиперпараметр, его задает человек.

**3. Алгоритм Ллойда.** Найти центры с наименьшей инерцией перебором невозможно, поэтому используют два чередующихся шага:

0. выбрать $k$ начальных центров;
1. **назначение:** каждый объект отнести к ближайшему центру;
2. **пересчет:** каждый центр заменить средним арифметическим своих объектов (отсюда «k средних»);
3. повторять шаги 1–2, пока назначения меняются.

Почему алгоритм всегда останавливается? Шаг назначения не увеличивает $J$: каждый объект выбирает ближайший центр. Шаг пересчета тоже не увеличивает $J$: среднее – это точка, до которой сумма квадратов расстояний наименьшая. Инерция не растет, а вариантов разбиения конечное число, значит, разбиения не могут повторяться бесконечно.

**4. Локальные минимумы.** Алгоритм останавливается, когда ни один шаг больше ничего не улучшает, но это не обязательно лучшее разбиение. Если два начальных центра попали в одно скопление, а в двух соседних скоплениях центра не оказалось, алгоритм застрянет: один центр «разрывается» между двумя скоплениями. Лекарство – **повторные запуски** с разными начальными центрами и выбор результата с наименьшей инерцией. В sklearn это параметр `n_init`.

**5. Инициализация k-means++.** Начальные центры лучше выбирать далеко друг от друга:

- первый центр – случайный объект;
- каждый следующий – случайный объект, причем объект $x$ выбирается с вероятностью, пропорциональной $D(x)^2$ – квадрату расстояния до ближайшего из уже выбранных центров.

Далекие от выбранных центров объекты (то есть объекты еще «не занятых» скоплений) выбираются чаще, а объекты рядом с уже выбранным центром – почти никогда. Так устроен `KMeans` в sklearn по умолчанию.

**6. Сколько кластеров?** Инерция всегда падает с ростом $k$: при $k = n$ каждый объект сам себе центр, и $J = 0$. Поэтому искать минимум инерции бессмысленно. **Метод «локтя»:** строим график инерции от $k$ и ищем точку, после которой падение резко замедляется, – «локоть». До него новый центр разделяет настоящие скопления, после него – дробит уже найденные. Локоть виден не всегда: на реальных данных график бывает плавным.

**7. Масштаб признаков.** Метод k средних считает евклидовы расстояния, как kNN (Б1.2). Признак с большими числами «перекрикивает» остальные, поэтому признаки стандартизируют (Б1.6).

**8. Ограничения.** Метод находит кластеры-«шары» примерно одинакового размера: граница между двумя кластерами – это прямая (плоскость) посередине между центрами. Вытянутые, изогнутые (например, «полумесяцы») и сильно разные по размеру скопления он делит неправильно. Среднее чувствительно к выбросам. Число $k$ нужно задать заранее. Методы без этих ограничений – на следующем занятии (Б2.4).

**Где почитать.** Я. Гудфеллоу, И. Бенджио, А. Курвилль «Глубокое обучение», раздел 5.8.2 «Кластеризация методом k средних».

In [ ]:
# Готовая ячейка: библиотеки, данные и помощники
import math
import random
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine     # только для загрузки набора Wine

RANDOM_STATE = 42
BLUE, NAVY, GREY, DARK_BLUE = '#4C9BFF', '#0F1B2D', '#969CA3', '#1F63C9'
COLORS = ['#4C9BFF', '#0F1B2D', '#969CA3', '#1F63C9', '#9CC8FF', '#5B6472', '#C9DEFF', '#283343', '#E1E3E5', '#7AB4FF']

def make_scene(seed=RANDOM_STATE):
    """Вид сверху на парковку: точки лидара от шести объектов, координаты в метрах.
    Возвращает точки X и номера объектов y. Номера объектов алгоритму не показываем –
    они нужны только нам, чтобы проверить результат."""
    rnd = random.Random(seed)
    objects = [((3, 3), (0.8, 0.5), 50), ((9, 2.5), (0.6, 0.6), 35), ((5, 9), (1.1, 0.6), 60),
               ((13, 9), (0.5, 0.9), 40), ((16, 3), (0.7, 0.7), 45), ((10, 13), (0.9, 0.5), 50)]
    X, y = [], []
    for c, ((cx, cy), (sx, sy), n) in enumerate(objects):
        for _ in range(n):
            X.append([round(rnd.gauss(cx, sx), 2), round(rnd.gauss(cy, sy), 2)])
            y.append(c)
    idx = list(range(len(X)))
    rnd.shuffle(idx)
    return [X[i] for i in idx], [y[i] for i in idx]

def standardize(X):
    """Стандартизация каждого столбца (Б1.6): среднее 0, стандартное отклонение 1."""
    n = len(X)
    result = [list(r) for r in X]
    for j in range(len(X[0])):
        mu = sum(r[j] for r in X) / n
        sigma = math.sqrt(sum((r[j] - mu) ** 2 for r in X) / n)
        for r in result:
            r[j] = (r[j] - mu) / sigma
    return result

def plot_clusters(ax, X, labels, centers=None, title=''):
    """Готовая функция: рисует точки цветом кластера и центры крестиками."""
    for j in sorted(set(labels)):
        pts = [x for x, l in zip(X, labels) if l == j]
        ax.scatter([p[0] for p in pts], [p[1] for p in pts], s=12, color=COLORS[j % len(COLORS)])
    if centers:
        ax.scatter([c[0] for c in centers], [c[1] for c in centers], marker='X', s=160,
                   color='white', edgecolor=NAVY, linewidth=2)
    ax.set_title(title); ax.set_aspect('equal'); ax.grid(alpha=0.3)

X_scene, y_scene = make_scene()
print('точек лидара:', len(X_scene), ' первая точка:', X_scene[0], '  ожидается 280')
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter([p[0] for p in X_scene], [p[1] for p in X_scene], s=12, color=GREY)
ax.set_title('Облако точек лидара: сколько здесь объектов?'); ax.set_xlabel('x, м'); ax.set_ylabel('y, м')
ax.set_aspect('equal'); ax.grid(alpha=0.3); plt.show()

**Задание С1. Шаг назначения.**

- `dist2(a, b)` – **квадрат** евклидова расстояния $\sum_d (a_d - b_d)^2$. Корень не нужен: ближайший центр по квадрату расстояния тот же, что и по расстоянию, а считать быстрее.
- `closest_center(x, centers)` – номер ближайшего к `x` центра. При равенстве расстояний – центр с меньшим номером (проверяйте строгим `<`).
- `assign_clusters(X, centers)` – список номеров ближайших центров для всех объектов.

*Считаем на бумаге.* Центры $(0, 0)$ и $(10, 10)$, точка $(4, 5)$: $d^2 = 16 + 25 = 41$ до первого и $36 + 25 = 61$ до второго – кластер 0.

In [ ]:
def dist2(a, b):
    return ...                                  # <-- сумма квадратов разностей координат

def closest_center(x, centers):
    best, best_d = 0, dist2(x, centers[0])
    for j in range(1, len(centers)):
        d = ...                                 # <-- квадрат расстояния до центра j
        if d < best_d:
            best, best_d = ...                  # <-- запомнить более близкий центр
    return best

def assign_clusters(X, centers):
    return ...                                  # <-- номер ближайшего центра для каждого объекта


print(dist2([4, 5], [0, 0]), dist2([4, 5], [10, 10]), '  ожидается 41 61')
print(closest_center([4, 5], [[0, 0], [10, 10]]), '  ожидается 0')
print(closest_center([5, 5], [[0, 0], [10, 10]]), '  ожидается 0 (ничья – меньший номер)')
toy = [[0, 0], [1, 0], [9, 9], [10, 9], [4, 5]]
print(assign_clusters(toy, [[0, 0], [10, 10]]), '  ожидается [0, 0, 1, 1, 0]')

**Задание С2. Шаг пересчета центров.**

`update_centers(X, labels, k, old_centers)` – для каждого кластера $j = 0, \dots, k-1$ собрать его объекты и вернуть их **покоординатное среднее**. Если в кластер не попал ни один объект (так бывает при неудачном старте), оставить старый центр `old_centers[j]`, иначе деление на ноль.

*Считаем на бумаге.* Кластер 0 из примера С1: точки $(0, 0)$, $(1, 0)$, $(4, 5)$, центр $\left(\frac{0 + 1 + 4}{3}; \frac{0 + 0 + 5}{3}\right) = \left(\frac{5}{3}; \frac{5}{3}\right)$.

In [ ]:
def update_centers(X, labels, k, old_centers):
    centers = []
    for j in range(k):
        points = ...                            # <-- объекты кластера j
        if points:
            ...                                 # <-- покоординатное среднее
        else:
            ...                                 # <-- пустой кластер: старый центр
    return centers


labels = assign_clusters(toy, [[0, 0], [10, 10]])
print([[round(v, 3) for v in c] for c in update_centers(toy, labels, 2, [[0, 0], [10, 10]])],
      '  ожидается [[1.667, 1.667], [9.5, 9.0]]')
print(update_centers(toy, [0, 0, 0, 0, 0], 2, [[0, 0], [7, 7]])[1], '  ожидается [7, 7]: пустой кластер сохранил центр')

**Задание С3. Инерция.**

`inertia(X, labels, centers)` – сумма квадратов расстояний от каждого объекта до центра **его** кластера (п. 2 теории). Чем меньше инерция, тем «плотнее» кластеры.

*Считаем на бумаге.* Точки $(0, 0)$ и $(2, 0)$ в одном кластере с центром $(1, 0)$: $J = 1 + 1 = 2$.

In [ ]:
def inertia(X, labels, centers):
    total = 0.0
    for x, label in zip(X, labels):
        total += ...                            # <-- квадрат расстояния до центра своего кластера
    return total


print(inertia([[0, 0], [2, 0]], [0, 0], [[1, 0]]), '  ожидается 2.0')
c2 = update_centers(toy, labels, 2, [[0, 0], [10, 10]])
print(round(inertia(toy, labels, [[0, 0], [10, 10]]), 3), '->', round(inertia(toy, labels, c2), 3),
      '  ожидается 45.0 -> 25.833: пересчет центров уменьшил инерцию')

**Вопрос С1–С3.** а) Почему для выбора ближайшего центра можно не извлекать корень? б) Докажите на примере, что среднее лучше любой другой точки: посчитайте сумму квадратов расстояний от чисел 1, 2, 6 до их среднего 3 и до числа 2. в) Почему при пересчете центров пустой кластер – это проблема и что еще, кроме «оставить старый центр», можно с ним сделать?

*Ваш ответ:*

**Задание С4. Алгоритм Ллойда.**

`kmeans(X, k, init_centers, max_iter=100)` – алгоритм из п. 3 теории:

1. начать с копии `init_centers` и `labels = None`;
2. в цикле: назначить кластеры; если назначения **не изменились** – остановиться; иначе запомнить их, пересчитать центры и добавить текущую инерцию в список `history`;
3. вернуть `centers, labels, history`.

Проверим на сцене с парковки. Начальные центры – шесть случайных точек облака: `random.Random(0).sample(X_scene, 6)`.

In [ ]:
def kmeans(X, k, init_centers, max_iter=100):
    centers = [list(c) for c in init_centers]
    labels = None
    history = []
    for _ in range(max_iter):
        new_labels = ...                        # <-- шаг назначения
        if new_labels == labels:
            ...                                 # <-- назначения не изменились – стоп
        labels = new_labels
        centers = ...                           # <-- шаг пересчета
        ...                                     # <-- инерция после шага
    return centers, labels, history


start = [list(x) for x in random.Random(0).sample(X_scene, 6)]
centers, labels, history = kmeans(X_scene, 6, start)
print('итераций:', len(history), '  инерция по шагам:', [round(h, 1) for h in history])
print('  ожидается: 6 итераций, от 2190.8 до 1046.2')
print('инерция не росла:', all(a >= b for a, b in zip(history, history[1:])), '  ожидается True')
print('размеры кластеров:', sorted(labels.count(j) for j in range(6)))

In [ ]:
# Готовая ячейка: как двигались центры и что получилось в итоге
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
c0 = start
plot_clusters(ax[0], X_scene, assign_clusters(X_scene, c0), c0, 'старт: шесть случайных точек')
c1 = update_centers(X_scene, assign_clusters(X_scene, c0), 6, c0)
plot_clusters(ax[1], X_scene, assign_clusters(X_scene, c1), c1, 'после первого шага')
plot_clusters(ax[2], X_scene, labels, centers, f'итог: инерция {history[-1]:.1f}')
plt.tight_layout(); plt.show()

**Вопрос С4.** а) Посмотрите на итоговую картинку: сколько настоящих объектов на парковке и сколько из них алгоритм нашел правильно? Что случилось с остальными? б) Алгоритм остановился и больше ничего не улучшит. Почему этот результат нельзя считать ответом? в) Почему остановка по условию «назначения не изменились» надежна, а условие «инерция стала меньше 0,001» было бы плохим?

*Ваш ответ:*

**Задание С5. Локальные минимумы и повторные запуски.**

- `random_init(X, k, rnd)` – $k$ **различных** случайных объектов в роли начальных центров: `rnd.sample(X, k)` (копии списков).
- `kmeans_restarts(X, k, n_init, init_fn, seed)` – запустить `kmeans` `n_init` раз со стартом `init_fn(X, k, rnd)`, где `rnd = random.Random(seed)` создается **один раз** перед циклом, и вернуть лучший результат `(centers, labels, inertia)` – с наименьшей итоговой инерцией.

Затем проведите эксперимент: 100 одиночных запусков со случайным стартом. Как часто алгоритм находит лучшее разбиение?

In [ ]:
def random_init(X, k, rnd):
    return ...                                  # <-- k различных случайных объектов

def kmeans_restarts(X, k, n_init, init_fn, seed=RANDOM_STATE):
    rnd = random.Random(seed)
    best = None
    for _ in range(n_init):
        centers, labels, history = ...          # <-- очередной запуск
        if best is None or history[-1] < best[2]:
            best = ...                          # <-- запомнить лучший
    return best


single = []
for s in range(100):
    c, l, h = kmeans(X_scene, 6, random_init(X_scene, 6, random.Random(s)))
    single.append(h[-1])
best_value = min(single)
share = sum(1 for v in single if v < best_value + 0.01) / len(single)
print(f'случайный старт: лучшая инерция {best_value:.1f}, найдена в {share:.0%} запусков, средняя {sum(single) / 100:.1f}')
print('  ожидается: 318.4, 43 %, 742.2')

centers_r, labels_r, J_r = kmeans_restarts(X_scene, 6, 10, random_init)
print(f'лучший из 10 запусков: {J_r:.1f}   ожидается 318.4')

In [ ]:
# Готовая ячейка: хороший и плохой результат рядом
bad = max(range(100), key=lambda s: single[s])
cb, lb, hb = kmeans(X_scene, 6, random_init(X_scene, 6, random.Random(bad)))
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
plot_clusters(ax[0], X_scene, labels_r, centers_r, f'лучший из 10 запусков: {J_r:.1f}')
plot_clusters(ax[1], X_scene, lb, cb, f'худший одиночный запуск: {hb[-1]:.1f}')
plt.tight_layout(); plt.show()

**Вопрос С5.** а) Чем отличается худший результат от лучшего? Опишите словами, где оказались центры. б) Вероятность найти лучшее разбиение за один запуск 0,43. Какова вероятность **не** найти его ни разу за 10 независимых запусков? в) Почему повторные запуски не помогают против ошибки из-за масштаба признаков?

*Ваш ответ:*

**Задание С6. Инициализация k-means++.**

`kmeans_pp_init(X, k, rnd)` по п. 5 теории:

1. первый центр – `rnd.choice(X)`;
2. пока центров меньше $k$: для каждого объекта посчитать $D(x)^2$ – квадрат расстояния до **ближайшего** из уже выбранных центров; вытянуть случайное число `r = rnd.random() * sum(D2)` и идти по объектам, накапливая сумму `acc`; первый объект, на котором `acc >= r`, становится новым центром.

Шаг 2 – это «рулетка»: каждый объект занимает на отрезке $[0, \sum D^2]$ кусок длиной $D(x)^2$, и случайная точка чаще падает на длинные куски. Объект, который уже стал центром, имеет $D = 0$ и выбран повторно быть не может.

In [ ]:
def kmeans_pp_init(X, k, rnd):
    centers = [list(rnd.choice(X))]
    while len(centers) < k:
        D2 = ...                                # <-- квадрат расстояния до ближайшего центра
        r = ...                                 # <-- случайная точка «рулетки»
        acc = 0.0
        for x, d in zip(X, D2):
            acc += ...                          # <-- накапливаем длины кусков
            if acc >= r:
                ...                             # <-- объект, на который упала точка
                break
    return centers


print(kmeans_pp_init([[0, 0], [0, 0.1], [10, 10]], 2, random.Random(1)), '  ожидается [[0, 0], [10, 10]]')

single_pp = []
for s in range(100):
    c, l, h = kmeans(X_scene, 6, kmeans_pp_init(X_scene, 6, random.Random(s)))
    single_pp.append(h[-1])
share_pp = sum(1 for v in single_pp if v < best_value + 0.01) / len(single_pp)
print(f'k-means++: лучшее найдено в {share_pp:.0%} запусков, средняя инерция {sum(single_pp) / 100:.1f}')
print('  ожидается: 69 %, 509.6 (у случайного старта 43 % и 742.2)')

**Вопрос С6.** а) В проверочном примере первым центром стала точка $(0, 0)$. Почему вторым почти наверняка станет $(10, 10)$, а не соседняя $(0; 0{,}1)$? Посчитайте ее «долю рулетки». б) k-means++ улучшил результат, но не до 100 %. Почему даже умный старт иногда проигрывает? в) Нужны ли повторные запуски, если мы уже используем k-means++?

*Ваш ответ:*

**Задание С7. Сколько объектов на парковке? Метод «локтя».**

Для каждого $k$ от 1 до 10 найдите лучшую инерцию из 10 запусков с k-means++ (`kmeans_restarts`) и сложите в список `inertias`. Затем посмотрите на график и найдите «локоть» (п. 6 теории). Для подсказки ячейка печатает, во сколько раз падает инерция при добавлении очередного кластера.

In [ ]:
ks = list(range(1, 11))
inertias = []
for k in ks:
    ...                                         # <-- лучшая инерция для этого k

for k, J in zip(ks, inertias):
    ratio = '' if k == 1 else f'   в {inertias[k - 2] / J:.1f} раза меньше, чем при k = {k - 1}'
    print(f'k = {k:2d}   инерция {J:8.1f}{ratio}')
print('  ожидается: k = 5: 911.0;  k = 6: 318.4 (в 2.9 раза меньше);  k = 7: 252.9 (всего в 1.3 раза)')

In [ ]:
# Готовая ячейка: «локоть»
plt.figure(figsize=(7, 3.6))
plt.plot(ks, inertias, 'o-', color=BLUE)
plt.axvline(6, color=NAVY, linestyle='--', label='локоть: k = 6')
plt.xlabel('k – число кластеров'); plt.ylabel('инерция'); plt.title('Метод «локтя»')
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Вопрос С7.** а) Почему инерция при $k = 7$ все равно меньше, чем при $k = 6$, хотя объектов шесть? Что делает седьмой центр? б) Чему равна инерция при $k = 280$? в) Предложите способ выбрать $k$, если «локтя» на графике не видно.

*Ваш ответ:*

**Задание С8. Wine без ответов: найдет ли метод три сорта?**

Спрячем от алгоритма сорта вина и попросим разбить 178 вин на 3 кластера. Потом сравним кластеры с настоящими сортами.

- `contingency(labels, y, k, n_classes)` – **таблица сопряженности**: строка – кластер, столбец – сорт, в клетке – сколько вин этого сорта попало в этот кластер.
- `purity(table)` – **чистота**: для каждого кластера берем самый частый в нем сорт, складываем эти количества по всем кластерам и делим на число объектов. Чистота 1 – каждый кластер состоит из вин одного сорта.

Сравните кластеризацию исходных признаков и стандартизованных (п. 7 теории). Кластеры нумеруются произвольно: кластер 0 не обязан быть сортом 0 – поэтому и нужна таблица.

In [ ]:
def contingency(labels, y, k, n_classes):
    table = [[0] * n_classes for _ in range(k)]
    for label, cls in zip(labels, y):
        table[label][cls] += ...                # <-- еще одно вино сорта cls в кластере label
    return table

def purity(table):
    n = sum(sum(row) for row in table)
    return ...                                  # <-- сумма самых частых сортов по кластерам


print(contingency([0, 0, 1, 1, 1], [1, 1, 0, 1, 0], 2, 2), '  ожидается [[0, 2], [2, 1]]')
print(round(purity([[0, 2], [2, 1]]), 3), '  ожидается 0.8')

wine = load_wine()
X_wine, y_wine = wine.data.tolist(), wine.target.tolist()
for name, Z in [('исходные признаки', X_wine), ('стандартизованные', standardize(X_wine))]:
    c, l, J = kmeans_restarts(Z, 3, 10, kmeans_pp_init)
    table = ...                                 # <-- таблица кластеры × сорта
    print(f'{name}: инерция {J:.1f}, чистота {purity(table):.3f}')
    for j, row in enumerate(table):
        print(f'   кластер {j}: сорт 0 – {row[0]:2d}, сорт 1 – {row[1]:2d}, сорт 2 – {row[2]:2d}')
print('  ожидается: исходные – чистота 0.702;  стандартизованные – чистота 0.966, инерция 1277.9')

In [ ]:
# Готовая ячейка: сверка с KMeans из sklearn на стандартизованных признаках
from sklearn.cluster import KMeans
sk = KMeans(n_clusters=3, n_init=10, random_state=0).fit(standardize(X_wine))
print(f'инерция sklearn: {sk.inertia_:.1f}   ожидается 1277.9 – та же, что у нашей реализации')

# Какие признаки «перекрикивают» остальные в исходных данных
spreads = sorted(((max(r[j] for r in X_wine) - min(r[j] for r in X_wine), wine.feature_names[j]) for j in range(13)), reverse=True)
for s, name in spreads[:3]:
    print(f'   размах признака {name}: {s:.1f}')

**Вопрос С8.** а) Почему на исходных признаках чистота всего 0,70? Какой признак определяет кластеры и почему? б) После стандартизации метод почти без ошибок нашел три сорта, хотя ответов не видел. О чем это говорит? в) Можно ли по высокой чистоте сказать, что кластеризация «правильная»? Вспомните пример Гудфеллоу с грузовиками и легковушками.

*Ваш ответ:*

In [ ]:
# Готовая ячейка: ограничение метода – «полумесяцы»
def make_moons(n=200, noise=0.08, seed=RANDOM_STATE):
    rnd = random.Random(seed)
    X, y = [], []
    for i in range(n):
        t = math.pi * rnd.random()
        if i % 2 == 0:
            X.append([math.cos(t) + rnd.gauss(0, noise), math.sin(t) + rnd.gauss(0, noise)]); y.append(0)
        else:
            X.append([1 - math.cos(t) + rnd.gauss(0, noise), 0.5 - math.sin(t) + rnd.gauss(0, noise)]); y.append(1)
    return X, y

X_moons, y_moons = make_moons()
cm, lm, Jm = kmeans_restarts(X_moons, 2, 10, kmeans_pp_init)
print(f'чистота на «полумесяцах»: {purity(contingency(lm, y_moons, 2, 2)):.3f}   ожидается 0.750')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
plot_clusters(ax[0], X_moons, y_moons, None, 'настоящие группы')
plot_clusters(ax[1], X_moons, lm, cm, 'k средних: граница – прямая')
plt.tight_layout(); plt.show()

**Вопрос С9.** Почему метод k средних разрезал «полумесяцы» поперек? Какой должна быть граница, чтобы разделить их правильно, и почему k средних такую границу построить не может?

*Ваш ответ:*

---
### Задания повышенной сложности

1. **Сокращение числа цветов изображения** (переход к Теме 2 «Компьютерное зрение»). Каждый пиксель – точка $(R, G, B)$. Кластеризуйте цвета картинки `load_sample_image('china.jpg')` из sklearn на $k = 4, 8, 16$ кластеров и замените каждый пиксель центром его кластера. Чтобы не ждать долго, обучайте k средних на 2000 случайных пикселях, а назначайте кластеры всем пикселям уменьшенной вдвое картинки. Во сколько раз уменьшается объем информации о цвете, если хранить номер кластера вместо трех чисел?
2. **Беспилотник без разметки.** Кластеризуйте объекты на дороге из Б1.2 (скорость, длина, высота после нормализации) на 3 кластера. Какая чистота относительно классов «пешеход, велосипедист, автомобиль»? Какие два класса метод путает и почему?
3. **Метод k медиан.** Замените среднее в шаге пересчета на покоординатную медиану (Б1.6) и добавьте в сцену 10 далеких выбросов. Сравните, насколько сдвигаются центры у k средних и у k медиан.
4. **Устойчивость числа кластеров.** Для Wine «локоть» почти не виден. Для $k = 2, \dots, 6$ сделайте 20 бутстрэп-выборок (Б2.2), кластеризуйте каждую и посчитайте, как часто пары объектов попадают в один кластер в разных запусках. При каком $k$ разбиение устойчивее всего?
5. **Мини-пакеты.** На каждом шаге пересчитывайте центры не по всем объектам, а по 30 случайным, сдвигая центр к среднему пакета с весом $1/t$, где $t$ – сколько объектов этот центр уже видел. Сравните время и инерцию с обычным алгоритмом.

In [ ]:
# Место для заданий повышенной сложности